
# Restaurant Recommendation System using Content-Based Filtering

## Project Overview

This project builds a content-based restaurant recommendation system that suggests restaurants based on user preferences such as cuisine, budget, ratings, online delivery, table booking, popularity, and city.

The recommendation engine combines cuisine similarity using TF-IDF vectorization with popularity-based ranking to generate personalized restaurant recommendations.

## Objectives

- Build a personalized restaurant recommendation system.
- Recommend restaurants based on user preferences.
- Rank recommendations using cuisine similarity and customer popularity.
- Simulate a real-world recommendation engine similar to food delivery platforms.

## Workflow

1. Import Libraries
2. Load Dataset
3. Data Preparation
4. Text Vectorization
5. Build Recommendation Engine
6. Generate Recommendations
7. Conclusion


## Import Libraries

In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity









## Load Dataset

In [5]:
df = pd.read_csv("...../data/restaurant.csv")

      


## Data Preparation

In [7]:
recommendation_df = df[
    [
        "Restaurant Name",
        "Cuisines",
        "City",
        "Price range",
        "Aggregate rating",
        "Votes",
        "Has Online delivery",
        "Has Table booking"
    ]
].copy()

recommendation_df["Cuisines"] = recommendation_df["Cuisines"].fillna("")
recommendation_df["Votes"] = recommendation_df["Votes"].fillna(0)

recommendation_df = recommendation_df.dropna().reset_index(drop=True)

recommendation_df.head()

,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Votes,Has Online delivery,Has Table booking
0,Le Petit Souffle,"French, Japanese, Desserts",Makati City,3,4.8,314,No,Yes
1,Izakaya Kikufuji,Japanese,Makati City,3,4.5,591,No,Yes
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",Mandaluyong City,4,4.4,270,No,Yes
3,Ooma,"Japanese, Sushi",Mandaluyong City,4,4.9,365,No,No
4,Sambo Kojin,"Japanese, Korean",Mandaluyong City,4,4.8,229,No,Yes


## Text Vectorization

In [9]:
# Convert cuisine text into numerical vectors

tfidf_vectorizer = TfidfVectorizer(stop_words="english")

cuisine_matrix = tfidf_vectorizer.fit_transform(
    recommendation_df["Cuisines"]
)

## Build Recommendation Engine

In [11]:
def recommend_restaurants(
    cuisine_preference,
    max_price_range,
    min_rating=3.5,
    online_delivery="Yes",
    table_booking=None,
    min_votes=0,
    city=None,
    top_n=5
):
    """
    Generate personalized restaurant recommendations based on cuisine preference, budget, ratings, online delivery, table booking, and city.
    """

    # Apply user preference filters
    filtered_restaurants = recommendation_df[
        (recommendation_df["Price range"] <= max_price_range) &
        (recommendation_df["Aggregate rating"] >= min_rating) &
        (recommendation_df["Has Online delivery"] == online_delivery) &
        (recommendation_df["Votes"] >= min_votes)
    ].copy()

    # Optional filter: Table booking
    if table_booking is not None:
        filtered_restaurants = filtered_restaurants[
            filtered_restaurants["Has Table booking"] == table_booking
        ]

    # Optional filter: City
    if city is not None:
        filtered_restaurants = filtered_restaurants[
            filtered_restaurants["City"] == city
        ]

    # No matching restaurants
    if filtered_restaurants.empty:
        return "No restaurants match the given criteria."

    # Convert user cuisine into TF-IDF vector
    user_vector = tfidf_vectorizer.transform([cuisine_preference])

    # Convert filtered restaurant cuisines into vectors
    filtered_matrix = tfidf_vectorizer.transform(
        filtered_restaurants["Cuisines"]
    )

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        user_vector,
        filtered_matrix
    ).flatten()

    # Final ranking score
    filtered_restaurants["Final Score"] = (
        similarity_scores * 0.7 +
        (filtered_restaurants["Votes"] / filtered_restaurants["Votes"].max()) * 0.3
    )

    
    # Sort recommendations
    recommended_restaurants = (
        filtered_restaurants
        .sort_values(by="Final Score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # Start index from 1 instead of 0
    recommended_restaurants.index = recommended_restaurants.index + 1

    # Round the final score
    recommended_restaurants["Final Score"] = (
        recommended_restaurants["Final Score"].round(3)
    )

    return recommended_restaurants[
        [
            "Restaurant Name",
            "Cuisines",
            "City",
            "Price range",
            "Aggregate rating",
            "Votes",
            "Has Table booking",
            "Final Score"
        ]
    ]

## Example 1: Budget-Friendly North Indian Restaurants

In [13]:
recommend_restaurants(
    cuisine_preference="North Indian",
    max_price_range=2,
    min_rating=3.8,
    online_delivery="Yes",
    min_votes=100
)

,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Votes,Has Table booking,Final Score
1,Jain Chawal Wale,North Indian,New Delhi,1,3.8,301,No,0.709
2,Jai Vaishno Rasoi,North Indian,New Delhi,1,4.2,171,No,0.705
3,BarBQ,"Chinese, North Indian",Kolkata,2,4.2,5288,No,0.694
4,Saffron Mantra,"North Indian, Chinese",Secunderabad,2,4.4,494,Yes,0.545
5,Dilli 6 On Wheels,"North Indian, Chinese",Gurgaon,1,3.8,142,No,0.534


## Example 2: Premium Japanese Restaurants

In [15]:
recommend_restaurants(
    cuisine_preference="Japanese",
    max_price_range=4,
    min_rating=4.2,
    table_booking="Yes",
    min_votes=200
)

,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Votes,Has Table booking,Final Score
1,Hauz Khas Social,"Continental, American, Asian, North Indian",New Delhi,3,4.3,7931,Yes,0.300
2,Cho Gao - Crowne Plaza Abu Dhabi,"Thai, Japanese, Chinese, Indonesian, Vietnamese",Abu Dhabi,4,4.4,246,Yes,0.300
3,Big Brewsky,"Finger Food, North Indian, Italian, Continenta...",Bangalore,3,4.5,5705,Yes,0.216
4,Exotica,"Mughlai, North Indian, Chinese",Hyderabad,3,4.3,3374,Yes,0.128
5,Lodi - The Garden Restaurant,"European, Lebanese, Mediterranean",New Delhi,4,4.2,2549,Yes,0.096


## Example 3: Dessert Cafes

In [17]:
recommend_restaurants(
    cuisine_preference="Desserts Cafe",
    max_price_range=3,
    min_rating=3.5
)

,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Votes,Has Table booking,Final Score
1,Maison Des Desserts,"Cafe, Desserts",New Delhi,3,3.9,552,No,0.717
2,Cinnabon,"Desserts, Cafe",New Delhi,1,4.1,397,No,0.712
3,Red Mango,"Cafe, Desserts",Gurgaon,2,3.6,365,No,0.711
4,Cinnabon,"Desserts, Cafe",New Delhi,1,3.9,360,No,0.711
5,Red Mango,"Cafe, Desserts",New Delhi,2,3.7,185,No,0.706


# Conclusion

## Summary

This project developed a **content-based restaurant recommendation system** that recommends restaurants based on user preferences such as cuisine, budget, ratings, online delivery, city, and table booking availability.

The recommendation engine combines **TF-IDF vectorization** with **cosine similarity** to measure cuisine similarity and incorporates customer popularity using vote counts to improve recommendation quality.

## Key Features

- Personalized restaurant recommendations
- Content-based filtering using TF-IDF
- Cosine similarity for cuisine matching
- Popularity-aware ranking using customer votes
- Support for multiple filtering options such as city, budget, ratings, and online delivery

## Real-World Applications

- Food delivery platforms (Zomato, Swiggy, Uber Eats)
- Restaurant discovery applications
- Personalized dining assistants
- Travel and tourism recommendation systems

## Future Improvements

- Hybrid recommendation systems combining content-based and collaborative filtering
- Personalized recommendations using user interaction history
- Location-aware recommendations using GPS
- Web deployment using Streamlit or Flask